# EFSM Phase 3 — QLoRA Fine-tuning (Google Colab)

**Runtime → Change runtime type → T4 GPU**

Before running, add these secrets (🔑 icon in left sidebar):
- `HF_TOKEN` — HuggingFace write token
- `WANDB_API_KEY` — Weights & Biases API key

In [ ]:
# Cell 1 — Secrets
from google.colab import userdata
import os

os.environ['HF_TOKEN']      = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

print('Authenticated.')

In [ ]:
# Cell 2 — Clone repo and install
import subprocess, os

REPO_URL = 'https://github.com/tasbidrahman10/empathetic-voice-llm.git'
REPO_DIR = 'empathetic-voice-llm'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('Requirements installed.')

In [ ]:
# Cell 3 — Download JSONL data from HuggingFace Hub
from huggingface_hub import hf_hub_download
import os

os.makedirs('data', exist_ok=True)
CHECKPOINT_REPO = 'tasbid001/efsm-checkpoints'

for split in ['train', 'val', 'test']:
    dest = f'data/{split}.jsonl'
    if not os.path.exists(dest):
        hf_hub_download(
            repo_id=CHECKPOINT_REPO,
            filename=f'data/{split}.jsonl',
            repo_type='model',
            local_dir='.',
            token=os.environ['HF_TOKEN'],
        )
        print(f'Downloaded {dest}')
    else:
        print(f'{dest} already present')

for split in ['train', 'val', 'test']:
    count = sum(1 for _ in open(f'data/{split}.jsonl'))
    print(f'{split}: {count} conversations')

In [ ]:
# Cell 4 — Train
# Colab gives ONE T4 — no DataParallel, no cross-device issues.
# CUDA_VISIBLE_DEVICES=0 is already set inside train.py.
import subprocess, os

env = os.environ.copy()
env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

result = subprocess.run(
    ['python', 'src/training/train.py', '--config', 'configs/config.yaml'],
    check=True,
    env=env,
)
print('Training complete. Return code:', result.returncode)

In [ ]:
# Cell 5 — Verify checkpoint on HuggingFace Hub
from huggingface_hub import list_repo_files
import os

files = list(list_repo_files('tasbid001/efsm-checkpoints', token=os.environ['HF_TOKEN']))
print('Files on HF Hub:')
for f in sorted(files):
    print(' ', f)